# Sparse Federated Learning Demo

This notebook is a lightweight, public-facing demonstration of the core ideas used in the original CIFAR-100 federated learning project.

The original project compared centralized training, standard Federated Averaging, sparse Federated Averaging and a simple secure aggregation mechanism. This demo avoids large model checkpoints and heavy training runs, and focuses on the reusable mechanics:

- IID and non-IID client partitioning;
- FedAvg-style aggregation;
- sparse update masking;
- zero-sum secure aggregation.

The full CIFAR-100 experiments are computationally heavier and are represented in the repository through the reusable source modules and project summary.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)


## 1. Simulating client label distributions

In [ ]:
n_samples = 1000
n_classes = 10
n_clients = 20

labels = np.repeat(np.arange(n_classes), n_samples // n_classes)
rng.shuffle(labels)

def iid_partition(labels, n_clients):
    indices = np.arange(len(labels))
    rng.shuffle(indices)
    return np.array_split(indices, n_clients)

def non_iid_partition(labels, n_clients, classes_per_client=2):
    client_indices = []
    all_classes = np.unique(labels)

    for _ in range(n_clients):
        selected_classes = rng.choice(all_classes, size=classes_per_client, replace=False)
        idx = np.where(np.isin(labels, selected_classes))[0]
        client_indices.append(rng.choice(idx, size=len(labels) // n_clients, replace=False))

    return client_indices

iid_clients = iid_partition(labels, n_clients)
non_iid_clients = non_iid_partition(labels, n_clients, classes_per_client=2)

def class_histogram(client_indices):
    return np.array([np.bincount(labels[idx], minlength=n_classes) for idx in client_indices])

iid_hist = class_histogram(iid_clients)
non_iid_hist = class_histogram(non_iid_clients)

fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(non_iid_hist, aspect="auto")
ax.set_title("Example non-IID client label distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Client")
plt.show()


## 2. Federated averaging on toy model states

In [ ]:
def fedavg(states):
    return {
        key: sum(state[key] for state in states) / len(states)
        for key in states[0]
    }

client_states = [
    {"weight": torch.tensor([1.0, 2.0]), "bias": torch.tensor([0.5])},
    {"weight": torch.tensor([2.0, 3.0]), "bias": torch.tensor([1.0])},
    {"weight": torch.tensor([3.0, 4.0]), "bias": torch.tensor([1.5])},
]

avg_state = fedavg(client_states)
avg_state


## 3. Sparse update masking

In [ ]:
def topk_mask(tensor, keep_fraction=0.5):
    flat = tensor.abs().flatten()
    k = max(1, int(len(flat) * keep_fraction))
    threshold = torch.topk(flat, k).values.min()
    return (tensor.abs() >= threshold).float()

update = torch.tensor([0.01, -0.80, 0.30, 0.02, 1.20, -0.10])
mask = topk_mask(update, keep_fraction=0.5)
sparse_update = update * mask

print("Original update:", update)
print("Mask:", mask)
print("Sparse update:", sparse_update)


## 4. Zero-sum secure aggregation demo

In [ ]:
def secure_aggregate(client_states, client_ids, seed_offset=0):
    num_clients = len(client_states)

    noise_sum = {}
    noise_dicts = []

    for idx in range(num_clients - 1):
        cid = client_ids[idx]
        torch.manual_seed(seed_offset + int(cid))
        state = client_states[idx]
        noise = {k: torch.randn_like(v) for k, v in state.items()}
        noise_dicts.append(noise)

        for k, v in noise.items():
            noise_sum[k] = noise_sum.get(k, torch.zeros_like(v)) + v

    last_noise = {k: -v for k, v in noise_sum.items()}
    noise_dicts.append(last_noise)

    masked_states = []
    for state, noise in zip(client_states, noise_dicts):
        masked_states.append({k: state[k] + noise[k] for k in state})

    return fedavg(masked_states)

secure_avg_state = secure_aggregate(client_states, client_ids=[0, 1, 2])

for key in avg_state:
    print(key, torch.allclose(avg_state[key], secure_avg_state[key]))


## Takeaway

This demo shows the mechanics behind the project in a lightweight and reproducible way. The original project applies these ideas to CIFAR-100 image classification using a DINO ViT-S/16 backbone and simulated federated client partitions.
